# 第22章 好みを教える：RLHFとDPO

「どちらの答えが良いか」だけで、モデルの性格を変える

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：29組の「良い答え／悪い答え」のペアで、QwenをDPOで学習させる。その前にDPOの損失を数式からゼロから実装し、TRLの学習の最初の損失と一致することを確かめる
- **身につく数学・理論**：なぜSFTだけでは足りないのか ／ 報酬モデルとブラッドリー・テリーモデル ／ シグモイド関数 ／ RLHF（PPO）の全体像とKLペナルティ ／ DPO：報酬モデルなしで好みを直接学ぶ ／ 報酬ハッキング・安全性・アラインメント ／ 検証可能な報酬による強化学習（推論モデルへの道）
- **新しく使うPython・PyTorch**：`F.logsigmoid` ／ `trl` の `DPOTrainer` ／ 学習ログ（`log_history`）の読み方

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★★ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> 第18章のSFTは「お手本をまねる」学習でした。でも、良い答え方はお手本1つでは決まりません。「この2つの答えなら、こっちのほうが良い」という**比べた好み**のほうが、人間にとって作りやすく、しかも細かいニュアンスを伝えられます。この章では、ChatGPTを丁寧で安全なアシスタントにした **RLHF** の考え方を学び、その簡単で強力な後継 **DPO** を、損失関数からゼロから実装して使います。

> **🔗 これまでの復習とつながり：第3章・第5章・第18章・第19章とのつながり**
>
> この章の理論と実装は、これまでの4つの章の道具をすべて合流させて作られています。
>
> - **第3章（softmax）**：生の点数の差を0〜1の確率に変える仕組み。この章の**シグモイド関数 $\sigma$** は、実は候補が2つだけのsoftmaxそのものです。
> - **第5章（交差エントロピーと $\log$）**：確率 $p$ に対する損失 $-\log p$、掛け算を足し算に・割り算を引き算に変える対数の性質、そして五分五分のときの損失 $-\log(0.5) = \log 2 \approx 0.693$ がそのままDPOの検算サインになります。
> - **第18章（SFT）**：「1つのお手本」をまねる学習。ここではその先にある「2つの答えを比べる学習」へ進みます。
> - **第19章（LoRA）**：元の重みを凍結して付せんだけ学習する仕組み。DPOでは「付せんを外した状態＝参照モデル $\pi_{\text{ref}}$」として1つのモデルで2役をこなせるため、無料GPUでも動きます！

**ミナト**：SFTでボンゴ先生になったのに、まだ何か足りないんですか？

**ボンゴ先生**：SFTは「100点のお手本」をまねる方法だ。でも世の中、100点のお手本を書くのは大変なのに、2つを比べて「こっちのほうがマシ」と言うのは簡単だろう？ ラーメンを自分で作るのは難しいけど、2杯食べ比べて「こっちがうまい」と言うのは誰でもできる。その「食べ比べの判定」から学ぶのが、この章の話だ。

## 好みのデータを見る

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：無料GPU（T4）推奨です。シグモイド・KL・DPO損失の検算セル（前半）は**CPUだけで数秒**で実行できます。後半の `DPOTrainer` 学習はT4でVRAM約5〜8GB・所要約1〜2分です。
> - **ダウンロード**：初回に `Qwen2.5-0.5B-Instruct`（約1GB）と好みデータ29組（数KB）を取得します。
> - **GPUが使えない日の代替**：CPUで全29組×3エポックのDPOを回すと数十分かかるため、GPUが借りられない日は前半の `dpo_loss` 検算までをCPUで確認し、後半は `prefs[:8]`・`num_train_epochs=1` に縮小するか、GPU確保後（またはKaggle T4）に実行してください。
> - **ライブラリ互換性**：`dtype=` は `transformers >= 4.56` 対応（推奨環境は `requirements-book.txt` を参照）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!pip install -q trl peft
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_prefs.jsonl
import json, os
if not os.path.exists("bongo_prefs.jsonl"):
    raise RuntimeError("bongo_prefs.jsonl のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
prefs = [json.loads(l) for l in open("bongo_prefs.jsonl", encoding="utf-8")]
print(len(prefs), "組")
p = prefs[11]
print("質問：", p["question"])
print("良い答え：", p["chosen"])
print("悪い答え：", p["rejected"])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
29 組
質問： 勉強のやる気が出ません
良い答え： やる気は、始める前ではなく、始めた後に出てくるものなんだ。まず5分だけ、コードを1行動かしてみよう。小さな成功が次の一歩を呼ぶ。やる気スイッチは、押すまで見つからない。スイッチを探すより、とりあえずスイッチオン！
悪い答え： やる気がないなら、やめればいいと思います。
```

</details>

同じ質問に対して、**選ばれた答え**（chosen）と **選ばれなかった答え**（rejected）の組です。悪い答えには、冷たい答え、間違った答え、役に立たない答えなどを入れてあります。このデータから、モデルに「良い答えを出しやすく、悪い答えを出しにくく」させます。

<details><summary><b>基礎の補足：SFTデータと好みデータ（Preference Dataset）の違い</b>（クリックで開く）</summary>

| 項目 | SFT（第18章） | 好み学習：RLHF / DPO（第22章） |
|---|---|---|
| 1件のデータ | `質問` ＋ `お手本（1つ）` | `質問` ＋ `良い答え（chosen）` ＋ `悪い答え（rejected）` |
| モデルに教えること | 「このお手本に近づけ」 | 「**chosen は出しやすく、rejected は出しにくく**」 |
| データ作成のしやすさ | 100点の文章を書くのが大変 | 2つの候補から良い方を選ぶだけなので集めやすい |
| 得意なこと | 出力形式・口調・基礎知識の型 | 丁寧さ・安全性・「やってはいけない答え」の抑制 |

</details>

#### 📝 確認問題（`ch22-q01` · Lv.1 · SFT（お手本模倣）と好み学習（比較判定）の違い）

第18章のSFT（教師あり微調整）に加えて、第22章で「好みのペア（chosen / rejected）」による学習を行う最大の理由はどれですか？

- **a.** 好みのペアデータで学習すると不要な重みが自動で刈り込まれ、モデルのパラメータ数を約半分に圧縮できるから
- **b.** SFTは英語のトークン列しか勾配計算できず、日本語の対話形式や敬語表現を1文字も学習できない制約があるから
- **c.** 100点のお手本を書くより2択の比較判定のほうが作りやすく、避けるべき悪い答えも明示的に押し下げられるから
- **d.** 好みのペアさえ集めれば、大量テキストによる事前学習（第13章）を完全に省いてゼロから高性能LLMを作れるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「完璧な料理をゼロから作る」のと「2皿を食べ比べて美味しい方を選ぶ」のたとえを思い出しましょう。
- **ヒント 2**：データ作成のしやすさに加え、SFTでは直接教えにくい「悪い答え（rejected）を避ける」対比の役割に注目してください。
- **正解**：`c`

**考え方**：SFTは「良い答えの確率を上げる」だけですが、好み学習では「良い答え（chosen）を上げ、悪い答え（rejected）を下げる」対比を学べるため、ニュアンス・丁寧さ・安全性の調整に非常に有効です。

1. SFT：質問と1つの正解（お手本）のペアを使う。何をしてはいけないかは直接教えにくい。
1. 好みデータ：同じ質問に対する良い答え（chosen）と悪い答え（rejected）のペアを使う。
1. 比較判定は人間にとって負担が小さく、モデルの細かい振る舞いの差を直接伝えられる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q01

</details>

#### 📝 確認問題（`ch22-q02` · Lv.2 · chosen / rejected ペアに含まれる情報の読み取り）

「勉強のやる気が出ません」という質問に対し、`chosen` が「まず5分だけコードを1行動かしてみよう…」、`rejected` が「やる気がないなら、やめればいいと思います。」のペアで学習させると、モデルにはどんな変化が期待されますか？

- **a.** 出力トークン数が少ないほど自動的に高評価になるため、常に1文だけの極端に短い答えを返すようになる
- **b.** `rejected` の文中に登場する「やる気」などの単語すべてが出力禁止となり、別の文脈でも二度と使えなくなる
- **c.** 学習前と比べて、励まして具体的な一歩を示す答えを出しやすくなり、突き放す冷たい答えを出しにくくなる
- **d.** 学習データ中の `chosen` の文章と1文字も違わない固定の文字列しか生成できなくなるよう上書きされる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：好み学習（DPO）が `chosen` と `rejected` それぞれの生成確率をどちら向きに調整するかを考えましょう。
- **ヒント 2**：丸暗記や特定単語の禁止ではなく、「励まして具体策を示す方針を上げ、突き放す方針を下げる」確率分布の変化に注目してください。
- **正解**：`c`

**考え方**：好みのペアを複数与えることで、モデルは「励ます・具体策を出す・親切」といった望ましい方針の確率を上げ、「突き放す・無責任」な方針の確率を下げていきます。

1. `chosen` の特徴：共感的・具体的・前向き。
1. `rejected` の特徴：冷たい・突き放す・役に立たない。
1. このペアの差を学ぶことで、似た質問に対しても前向きで具体的な応答を好むようになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q02

</details>

## 報酬モデルと、好みの確率

まずは、RLHFの考え方から見ていきましょう。

> **💡 理論のポイント：報酬モデル：答えに点数をつけるモデル**
>
> 人間の好みを学習させるには、まず「答えの良さ」を数にする必要があります。そこで、答えを入力すると**点数（報酬）**を出す **報酬モデル**（reward model）を作ります。
>
> 報酬モデルは、好みのペアから学習します。「良い答えの点数 $r_c$ が、悪い答えの点数 $r_r$ より高くなる」ようにするのです。そのとき使われるのが、**ブラッドリー・テリーモデル** という考え方です。
>
> $$P(\text{良い答えが選ばれる}) = \sigma(r_c - r_r)$$
>
> 点数の**差**をシグモイド関数 $\sigma$ に入れたものを、「良い答えが選ばれる確率」と考えます。スポーツのレーティング（強さの点数の差から勝率を予想する）と同じ発想です。

#### 📐 数学のポイント：シグモイド関数とブラッドリー・テリーモデル：9ステップで理由から理解する

**① 解決したい具体的な問題**：2つの答えについた「生の点数 $r_c$（良い答え）と $r_r$（悪い答え）」から、「良い答えが選ばれる確率（$0$ 〜 $1$）」を計算したい。

**② 素朴な方法では困る理由**：ただの差 $r_c - r_r$ はマイナスにも $+100$ にもなるので確率になりません。かといって $r_c / (r_c + r_r)$ と割ると、点数が負のときに分母が $0$ になったり確率が $1$ を超えたりして破綻します。

**③ 新しい発想**：点数の差 $x = r_c - r_r$ を、どんな実数でも滑らかに $0$ と $1$ のあいだに押し込むS字の関数 **シグモイド関数 $\sigma(x)$** に通す。

**④ 記号の1つ1つの読み方**：
$$\sigma(x) = \frac{1}{1 + e^{-x}}, \qquad P(y_c \succ y_r) = \sigma(r_c - r_r)$$
- $\sigma$（シグマ）：シグモイド関数（ギリシャ文字の小文字。合計の $\sum$ は大文字なので区別しよう）
- $e$：ネイピア数（約 $2.718$、第3章のsoftmaxで登場）
- $r_c, r_r$：良い答え（chosen）と悪い答え（rejected）の報酬スコア
- $y_c \succ y_r$：「$y_c$ のほうが $y_r$ より好まれる」という記号

**⑤ 小さな数字で手計算**：
- 同点（$r_c = 1, r_r = 1 \implies x = 0$）：$\sigma(0) = \dfrac{1}{1 + e^0} = \dfrac{1}{1 + 1} = 0.500$（ちょうど五分五分！）
- 差が $+2$（$r_c = 3, r_r = 1 \implies x = 2$）：$e^{-2} \approx 0.135$ なので $\sigma(2) = \dfrac{1}{1 + 0.135} \approx 0.881$（約88%で良い答えが勝つ）
- 差が $-2$（$r_c = 1, r_r = 3 \implies x = -2$）：$e^{+2} \approx 7.389$ なので $\sigma(-2) = \dfrac{1}{1 + 7.389} \approx 0.119$（対称性 $\sigma(-x) = 1 - \sigma(x)$ に注目！）

**⑥ 第3章 softmax との完全な一致（2択のsoftmax＝シグモイド）**：
候補が $r_c$ と $r_r$ の2つだけのsoftmaxを、分子・分母とも $e^{r_c}$ で割ってみましょう。
$$\frac{e^{r_c}}{e^{r_c} + e^{r_r}} = \frac{1}{1 + e^{r_r - r_c}} = \frac{1}{1 + e^{-(r_c - r_r)}} = \sigma(r_c - r_r)$$
なんと**ブラッドリー・テリーモデルは、2候補のsoftmaxと数学的にまったく同じ式**なのです！

**⑦ Pythonで確かめる**：

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import math
for x in [-4, -2, 0, 2, 4]:
    print(f"σ({x:+d}) = {1 / (1 + math.exp(-x)):.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
σ(-4) = 0.018
σ(-2) = 0.119
σ(+0) = 0.500
σ(+2) = 0.881
σ(+4) = 0.982
```

</details>

**⑧ 極端なときの振る舞い**：差 $x \to +\infty$ で $\sigma(x) \to 1$（ほぼ確実に良い答えが選ばれる）、$x \to -\infty$ で $\sigma(x) \to 0$、$x = 0$ でちょうど $0.5$。また両方に $+100$ を足しても差 $r_c - r_r$ は変わらない（平行移動不変）。

**⑨ 学習とのつながり**：報酬モデルを学習するときは、この確率に第5章の負の対数（交差エントロピー）をつけた $-\log\sigma(r_c - r_r)$ を最小化します。この形が、すぐ次のDPOの式にそのまま現れます！

> **💡 理論のポイント：RLHF：報酬を最大にするように強化学習**
>
> 報酬モデルができたら、次はLLMを **強化学習**（reinforcement learning）で鍛えます。LLMに答えを生成させ、報酬モデルで点数をつけ、点数が高い答えを出しやすくなるようにパラメータを調整する。これをくり返します（よく使われる方法が **PPO**）。人間のフィードバック（Human Feedback）から学ぶ強化学習（RL）なので、**RLHF** と呼びます。
>
> ただし、報酬だけを追いかけると、モデルは**報酬モデルの穴をつく**ようになります。たとえば報酬モデルが「長い答え」を好むクセを持っていたら、意味のない長文を書くようになる。これを **報酬ハッキング** と言います。そこで、「元のモデル（参照モデル）からあまり離れすぎない」という罰（**KLペナルティ**）を加えます。
>
> $$\text{目標} = \text{報酬} - \beta \times \text{元のモデルからの離れ具合（KLダイバージェンス）}$$
>
> RLHFはとても強力ですが、報酬モデル・強化学習・生成をぐるぐる回すので、仕組みが複雑で、学習も不安定になりがちです。

#### 📐 数学のポイント：KLダイバージェンス $\mathrm{KL}(p \,\|\, q)$：9ステップで理由から理解する

**① 解決したい具体的な問題**：学習中のモデルが出す確率分布 $p$ が、元の参照モデルの確率分布 $q$ から「どれくらいズレたか」を1つの数値（罰金）で測りたい。

**② 素朴な方法では困る理由**：単純に差を足す $\sum_i (p_i - q_i)$ は、$\sum p_i = 1, \sum q_i = 1$ なので $1 - 1 = 0$ となり、どれだけズレても常に0で測れません。また単純な差の絶対値だと、元のモデルが「ほとんど起きない（$q_i = 0.0001$）」と判断していた変な単語に確率を移したときの危険度（何倍になったか）を十分重く罰せられません。

**③ 新しい発想**：各選択肢について「元の確率の**何倍**になったかの対数 $\log(p_i / q_i)$」をとり、それを今のモデルが実際に選ぶ確率 $p_i$ で重みづけ平均する。

**④ 記号の1つ1つの読み方**：
$$\mathrm{KL}(p \,\|\, q) = \sum_i p_i \log \frac{p_i}{q_i}$$
- $p$：今のモデル（学習中の方策 $\pi$）の確率分布
- $q$：元のモデル（参照モデル $\pi_{\text{ref}}$）の確率分布
- $\|$：2つの分布を比べる区切り線（順番に意味があり、$\mathrm{KL}(p\|q) \neq \mathrm{KL}(q\|p)$ の**非対称**！）
- $\log\dfrac{p_i}{q_i}$：第5章と同じ自然対数。$p_i = q_i$ なら $\log 1 = 0$、$p_i > q_i$ ならプラス、$p_i < q_i$ ならマイナス

**⑤ 小さな数字で手計算**：
元のモデルが次のトークン「A・B」を $q = (0.5, 0.5)$ で選び、今のモデルが $p = (0.9, 0.1)$ に偏ったとき：
$$\mathrm{KL}(p \,\|\, q) = 0.9 \log\frac{0.9}{0.5} + 0.1 \log\frac{0.1}{0.5} \approx 0.9 \times 0.588 + 0.1 \times (-1.609) \approx 0.529 - 0.161 = 0.368$$
もし $p = q = (0.5, 0.5)$（ズレなし）なら、$0.5\log 1 + 0.5\log 1 = 0$ です。

**⑥ Pythonでの最小コード**：
```python no-run
import math
p, q = [0.9, 0.1], [0.5, 0.5]
kl = sum(pi * math.log(pi / qi) for pi, qi in zip(p, q) if pi > 0)
print(round(kl, 3))   # 0.368
```

**⑦ 絶対に押さえる2つの性質**：
1. **必ず0以上**（$\mathrm{KL}(p\|q) \ge 0$）：各項にマイナスが出ても、合計すると必ず0以上になり、0になるのは $p = q$ のときだけ。
2. **第5章の交差エントロピーの仲間**：$\sum p_i \log\frac{p_i}{q_i} = \big(-\sum p_i \log q_i\big) - \big(-\sum p_i \log p_i\big)$（交差エントロピー − エントロピー）。PyTorchでは `F.kl_div(log_q, p, reduction="batchmean")` で計算できます（第1引数に対数 $\log q$、第2引数に確率 $p$ を渡す逆順の仕様なので注意）。

**⑧ $\beta$ の効き方**：RLHFの目標 $\mathbb{E}[r(x,y)] - \beta\,\mathrm{KL}(\pi \,\|\, \pi_{\text{ref}})$ で、$\beta$ が大きすぎるとモデルが元のまま動かなくなり、$\beta$ が小さすぎると報酬ハッキングで文章が壊れます。

**⑨ DPOへの橋渡し**：実は、この「KLペナルティつきの目標」があるからこそ、次の節で魔法のような数学変形が可能になります！

#### 📝 確認問題（`ch22-q03` · Lv.2 · ブラッドリー・テリーモデルとシグモイド関数の手計算）

ブラッドリー・テリーモデル $P(\text{良い答えが選ばれる}) = \sigma(r_c - r_r)$ を使います。良い答えの報酬が $r_c = 3$、悪い答えの報酬が $r_r = 1$ のとき、良い答えが選ばれる確率はおよそいくつですか？（$e^{-2} \approx 0.135$ として小数で回答、誤差 $\pm 0.02$ まで正解）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：ブラッドリー・テリーモデルでは報酬の絶対値ではなく「差」を求め、シグモイド関数 $\sigma(x) = \frac{1}{1+e^{-x}}$ に入れます。
- **ヒント 2**：点数の差は $r_c - r_r = 3 - 1 = 2$ です。問題文の $e^{-2} \approx 0.135$ を使って $\frac{1}{1 + e^{-2}}$ を割り算しましょう。
- **正解**：`0.881`

**考え方**：ブラッドリー・テリーモデルでは報酬の絶対値ではなく「差」$r_c - r_r$ をシグモイド関数に通して勝率（選ばれる確率）を出します。

1. 報酬の差を計算する：$r_c - r_r = 3 - 1 = 2$。
1. シグモイド関数に入れる：$\sigma(2) = \dfrac{1}{1 + e^{-2}}$。
1. $e^{-2} \approx 0.135$ なので、$\dfrac{1}{1 + 0.135} \approx 0.881$（約88.1%）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q03

</details>

#### 📝 確認問題（`ch22-q04` · Lv.2 · 報酬ハッキングとKLペナルティの役割）

RLHFで、報酬モデルが「長い答えほど高い点数をつける」というクセを持っていたとします。KLペナルティなしで報酬だけを最大化すると何が起きやすく、それをどう防ぎますか？

- **a.** モデルが答えを1文字だけ出力して即終了するようになり、生成時の温度（temperature）を0に固定すれば防げる
- **b.** 中身のない冗長な長文を書くようになる（報酬ハッキング）。参照モデルから離れすぎないKLペナルティ等で防ぐ
- **c.** すべての重みパラメータが0に収束して出力が空白になるため、学習率を100倍に大きく引き上げれば簡単に防げる
- **d.** 強化学習の収束時間が約半分に短縮されて応答品質も自然に向上するだけで、実運用上の問題は特に起きない

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：報酬モデルの採点のクセ（抜け穴）を強化学習が過剰に突いてしまう現象と、それを抑えるブレーキ項を思い出しましょう。
- **ヒント 2**：目標式 $\text{報酬} - \beta \times \mathrm{KL}(\pi \| \pi_{\text{ref}})$ の第2項が「元のモデルから離れすぎない制約」である点に注目してください。
- **正解**：`b`

**考え方**：報酬モデルは人間の好みの近似にすぎないため、無制限に報酬を追うと抜け穴（長文・過剰な美辞麗句・同じ単語の繰り返しなど）を突く「報酬ハッキング」が起きます。KLペナルティは参照モデルという「常識の錨（いかり）」につなぎ止める役割を果たします。

1. 報酬モデルのクセ（例：長さバイアス）を、強化学習が容赦なく利用してスコアを稼ぎ始める。
1. 結果として、人間には読みにくい無意味な長文が増える（報酬ハッキング）。
1. 対策：目標から $\beta \cdot \mathrm{KL}(\pi \| \pi_{\text{ref}})$ を引いて元の分布から離れすぎないようにし、報酬データも改善する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q04

</details>

*(この節の追加問題 `ch22-q05`, `ch22-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html で自動採点・復習できます)*

## DPO：報酬モデルなしで、好みを直接学ぶ

2023年に、スタンフォード大学の研究者たちが、すごいことに気づきました。上のRLHFの「目標」を数学的に変形すると、**報酬モデルも強化学習も使わずに**、好みのペアから直接LLMを学習させる損失関数が導けるのです。それが **DPO**（Direct Preference Optimization）です。

#### 📐 数学のポイント：DPOの目的関数 $L_{\text{DPO}}$：9ステップで記号の1つ1つから完全に理解する

**① 解決したい具体的な問題**：RLHFでは「①報酬モデルの学習 → ②PPO強化学習」の2段階が必要で、モデルも複数要り、学習が非常に不安定です。報酬モデルを作らずに、好みのペア `(質問, chosen, rejected)` だけから直接LLMを学習させたい！

**② 素朴な方法ではなぜダメなのか？**
もし単純に「良い答えの対数確率を上げ、悪い答えを下げる」だけの損失 $-\log\sigma\big(\beta(\log\pi(y_c) - \log\pi(y_r))\big)$ を使うと、次の**2つの致命的な欠陥**が起きます。
1. **ブレーキがない**：元のモデルから際限なく離れてしまい、日本語として壊れた文章になりやすい。
2. **長さバイアスに負ける**：文章全体の対数確率は各トークンの負の対数確率の合計なので、**長い良い答え**（例：`-60`）より**短い悪い答え**（例：`-25`）のほうが最初から圧倒的に値が高い！そのまま引き算すると、最初から `-60 - (-25) = -35` という巨大なハンデを背負ってしまいます。

**③ RLHF→DPOの導出の肝（どんな仮定・簡略化をしているか）**：
RLHFのKL制約つき目標 $\max_\pi \mathbb{E}[r(x,y)] - \beta\,\mathrm{KL}(\pi \,\|\, \pi_{\text{ref}})$ を数学的に解くと、最適方策 $\pi^*$ は次の**閉じた形（解析解）**で表せることが知られています。
$$\pi^*(y \mid x) = \frac{1}{Z(x)}\,\pi_{\text{ref}}(y \mid x)\exp\!\left(\frac{1}{\beta}\,r(x, y)\right)$$
ここで $Z(x)$ は確率の合計を1にするための正規化定数（全文章の和なので直接は計算不能）です。ところが、この式を $r(x, y)$ について逆算すると、
$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$
となります！これをブラッドリー・テリーモデルの差 $r_c - r_r$ に代入すると、計算できない厄介者 **$\beta \log Z(x)$ が引き算で完全に消えて（相殺されて）しまう**のです！
> **導出における簡略化・前提の整理**：この導出は「①人間の好みがブラッドリー・テリーモデルに従う」「②KL制約つきRLHFの厳密な最適方策 $\pi^*$ によって報酬を言い換えられる」「③固定されたオフラインの好みペア上で $\pi$ を直接更新してもよい（RLHFのように学習途中のモデル自身から再サンプリングするオンポリシー分布ズレを近似・省略する）」という仮定に基づいています。だからこそ、強化学習ループなしの単純な教師あり風の損失関数に落とし込めるのです。

**ミナト**：おおっ、計算できない $Z(x)$ が引き算で消えるから、報酬モデルなしで直接書けるんですね！ じゃあ実際のDPOの式を記号1つずつ読んでみたいです。

**ボンゴ先生**：よし、ここからは1つも記号を飛ばさずに、読み方・意味・具体的な数字の手計算まで一気に確かめよう！

#### 📐 数学のポイント：DPOの式と記号・手計算トレース

**④ DPO目的関数の記号を1つ残らず読む**：
$$L_{\text{DPO}} = -\log \sigma\!\Big(\beta \big[ (\log \pi(y_c) - \log \pi_{\text{ref}}(y_c)) - (\log \pi(y_r) - \log \pi_{\text{ref}}(y_r)) \big]\Big)$$

| 記号 | 読み方・正体 | 絶対に間違えないためのポイント |
|---|---|---|
| $\pi$ | **学習中のモデル（方策、policy）** | **円周率 $3.14$ ではない！** 英語 policy の p にあたるギリシャ文字 |
| $\pi_{\text{ref}}$ | **参照モデル（reference）** | 学習開始前の重みを凍結した元のLLM（ブレーキ役・出発点の基準） |
| $y_c, y_r$ | 良い答え（chosen）・悪い答え（rejected） | 同じ質問 $x$ に対する比較ペアの文章 |
| $\log \pi(y_c)$ | 良い答え全体の対数確率 | 答えの全トークンの $\log P_\pi(\text{token}_t \mid \text{前})$ の合計（0以下の負の数） |
| $\Delta_c = \log\pi(y_c) - \log\pi_{\text{ref}}(y_c)$ | 良い答えの「書きやすさの変化」 | $\log(\pi/\pi_{\text{ref}})$。元より書きやすくなると**プラス** |
| $\Delta_r = \log\pi(y_r) - \log\pi_{\text{ref}}(y_r)$ | 悪い答えの「書きやすさの変化」 | 元より書きにくくなると**マイナス**（減らしたい！） |
| $\beta$ | KLペナルティの強さ（例：`0.1`） | 大きいほど参照モデルから離れにくくなる |
| $\sigma(\cdot)$ | シグモイド関数 | マージン $m = \beta(\Delta_c - \Delta_r)$ を確率 $(0, 1)$ に変換 |
| $-\log(\cdot)$ | 負の対数（第5章の交差エントロピー） | 確率が1に近いほど損失0、確率0.5なら $\log 2 \approx 0.693$ |

**⑤ なぜわざわざ $\log\pi_{\text{ref}}$ を引き算するのか？（2つの本質的理由）**：
1. **暗黙のKLペナルティが自動で入る**：$\beta(\log\pi - \log\pi_{\text{ref}}) = \beta\log(\pi/\pi_{\text{ref}})$ という「元のモデルとの比」を暗黙の報酬にすることで、元のモデルの自然な日本語を壊さないブレーキが組み込まれる。
2. **出発点の長さバイアスを打ち消せる**：良い答えが `-60`、悪い答えが `-25` でも、それぞれから参照モデルの値（`-60` と `-25`）を引くため、出発点はどちらも `0` にリセットされ、純粋な「変化の差」だけで公平に比べられる！

**⑥ 具体的な数字で手計算トレース（なぜ初期損失が常に $0.693$ なのか）**：
参照モデルで $\log\pi_{\text{ref}}(y_c) = -60$、$\log\pi_{\text{ref}}(y_r) = -25$、$\beta = 0.1$ とします。

| 状態 | $\Delta_c$ | $\Delta_r$ | マージン $m = 0.1(\Delta_c - \Delta_r)$ | 確率 $\sigma(m)$ | DPO損失 $-\log\sigma(m)$ |
|---|---|---|---|---|---|
| **学習開始直後（$\pi = \pi_{\text{ref}}$）** | $0$ | $0$ | **$0.0$** | **$0.500$** | **$-\log 0.5 = \log 2 \approx 0.693$** |
| 良い答えが $+10$ 書きやすく（`-50`） | $+10$ | $0$ | $+1.0$ | $0.731$ | $0.313$（減少！） |
| 悪い答えが $-10$ 書きにくく（`-35`） | $0$ | $-10$ | $+1.0$ | $0.731$ | $0.313$（同じく減少！） |
| 両方達成（`chosen +10` かつ `rejected -10`） | $+10$ | $-10$ | $+2.0$ | $0.881$ | $0.127$（大きく減少！） |
| 逆方向に失敗（`chosen -10`） | $-10$ | $0$ | $-1.0$ | $0.269$ | $1.313$（急上昇！） |

**⑦ 純粋Pythonでの実装イメージ**：
```python no-run
import math
def dpo_loss_simple(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    margin = beta * ((pi_c - ref_c) - (pi_r - ref_r))
    prob = 1.0 / (1.0 + math.exp(-margin))
    return -math.log(prob)
```
なおPyTorchでは、極端に負の `margin` で `math.exp(-margin)` がオーバーフローするのを防ぐため、数学的に等価かつ数値安全な `-F.logsigmoid(margin)` を使います。

**⑧ 注意点（DPOのクセ）**：式は差 $\Delta_c - \Delta_r$ だけを見るため、学習率が大きすぎると良い答えの確率 $\Delta_c$ まで一緒に下がりながら悪い答えがもっと下がる（`rewards/chosen` がマイナスになる）現象が起きることがあります。また長文バイアスも完全には消えないため、生成文の目視と長さの確認が大切です。

**⑨ 実際のPyTorchコードで検算しよう**：下のセルで、上の手計算テーブルの数字と小数第3位まで完全に一致することを確かめます！

式だけだと分かりにくいので、数で計算してみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=0.1):
    """各引数は、その答え全体の対数確率（トークンごとの log 確率の合計）"""
    margin = beta * ((pi_c - ref_c) - (pi_r - ref_r))
    return -F.logsigmoid(margin)

ref_c, ref_r = torch.tensor(-60.0), torch.tensor(-25.0)     # 学習前：悪い答え（短い）のほうが書きやすい
print("学習前（方策＝参照）：", dpo_loss(ref_c, ref_r, ref_c, ref_r).item())
print("良い答えが +10 書きやすく：", dpo_loss(ref_c + 10, ref_r, ref_c, ref_r).item())
print("悪い答えが -10 書きにくく：", dpo_loss(ref_c, ref_r - 10, ref_c, ref_r).item())
print("両方：", dpo_loss(ref_c + 10, ref_r - 10, ref_c, ref_r).item())

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
学習前（方策＝参照）： 0.6931471824645996
良い答えが +10 書きやすく： 0.3132616877555847
悪い答えが -10 書きにくく： 0.3132616877555847
両方： 0.12692801654338837
```

</details>

> **💡 理論のポイント：最初の損失は log 2**
>
> 学習を始めた瞬間は、方策＝参照なので、括弧の中は0、$\sigma(0) = 0.5$、損失は $-\log 0.5 = \log 2 \approx 0.693$ です。第5章で「最初の損失は $\log V$」を確かめたのと同じで、DPOの学習ログで最初に **0.693** が出ていたら、正しく始まっている印です。
>
> 注目してほしいのは、学習前の対数確率は「悪い答え（-25）」のほうが「良い答え（-60）」より高い、つまり**悪い答えのほうが書きやすい**状態だったことです。悪い答えは短いので、確率の掛け算の回数が少ないからです。DPOは、確率そのものではなく「元からの**変化**」を比べるので、出発点での長さの違いには惑わされません（ただし、学習を続けると長い答えを好むようになりやすいクセも知られています。第20章の評価で、答えの長さも見ておくと安心です）。

#### 📝 確認問題（`ch22-q07` · Lv.1 · DPO目的関数の記号（π, π_ref, y_c, y_r, β）の正確な読み方）

DPOの損失式 $L_{\text{DPO}} = -\log\sigma\Big(\beta\big[(\log\pi(y_c) - \log\pi_{\text{ref}}(y_c)) - (\log\pi(y_r) - \log\pi_{\text{ref}}(y_r))\big]\Big)$ に出てくる記号 $\pi$ と $\pi_{\text{ref}}$ の意味として正しいものはどれですか？

- **a.** $\pi$ は円周率 $3.14159\dots$ の正確な定数、$\pi_{\text{ref}}$ は計算高速化のため丸めた近似値 $3.14$ を表す
- **b.** $\pi$ は人間が選んだ良い答えのトークン列、$\pi_{\text{ref}}$ は比較対象となった悪い答えのトークン列を表す
- **c.** $\pi$ は学習中のモデル（方策、policy）、$\pi_{\text{ref}}$ は学習前の固定された参照モデル（reference）を表す
- **d.** $\pi$ は最適化に使う学習率の現在値、$\pi_{\text{ref}}$ はステップ開始時に設定した初期バッチサイズを表す

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：強化学習では、応答の確率を決めるモデル（行動ルール）のことを英語で何と呼ぶか思い出しましょう。
- **ヒント 2**：**policy（方策）** の頭文字 p に対応するギリシャ文字 $\pi$ と、基準点となる **reference（参照）** の添字に注目してください。
- **正解**：`c`

**考え方**：DPO式は見た目が長くて圧倒されがちですが、登場人物は「今のモデル $\pi$」「元のモデル $\pi_{\text{ref}}$」「良い答え $y_c$」「悪い答え $y_r$」「ブレーキ係数 $\beta$」の5つだけです。

1. $\pi$：今から重みを更新していく学習中のLLM（方策）。
1. $\pi_{\text{ref}}$：学習開始時の重みを凍結した参照LLM（基準点）。
1. $\log\pi(y_c)$：学習中のLLMが、良い答え $y_c$ を生成する対数確率の合計。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q07

</details>

#### 📝 確認問題（`ch22-q08` · Lv.2 · DPOの初期損失が必ず log 2 ≈ 0.693 になる理由）

DPOの学習を正しく開始した瞬間（ステップ1の前）の損失が、データの中身や $\beta$ の値に関係なく必ず $\log 2 \approx 0.693$ になるのはなぜですか？

- **a.** 開始時は方策と参照モデルが一致してマージンが $0$ となり、$-\log\sigma(0) = \log 2$ になるから
- **b.** PyTorchの `F.logsigmoid` が数値安定化の初期化として最初の1ステップ目だけ常に `-0.693` を返す仕様だから
- **c.** DPO学習ではモデルの語彙サイズ $V$ が2語だけに縮退しているため、初期交差エントロピーが $\log V = \log 2$ になるから
- **d.** ハイパーパラメータ $\beta$ を常に `0.693` に固定して損失の定数項として足し込む実装規約になっているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：学習開始の瞬間は、学習中のモデル $\pi$ と参照モデル $\pi_{\text{ref}}$ の重みがどうなっているか思い出しましょう。
- **ヒント 2**：$\pi = \pi_{\text{ref}}$ をDPOの損失式に代入し、マージンの値と $\sigma(0)$ の値を順に追ってみてください。
- **正解**：`a`

**考え方**：第5章の言語モデル学習で「初期損失 $= \log V$」が健全性チェックになったように、DPOでは「初期損失 $= \log 2 \approx 0.693$」が「方策と参照モデルが正しく同じ出発点に揃っている」黄金の検算サインです。

1. 学習開始時は $\pi = \pi_{\text{ref}}$ なので、$\log\pi(y_c) - \log\pi_{\text{ref}}(y_c) = 0$、$\log\pi(y_r) - \log\pi_{\text{ref}}(y_r) = 0$。
1. マージン $= \beta(0 - 0) = 0$。
1. $\sigma(0) = 0.5 = \dfrac{1}{2}$。
1. 損失 $= -\log(1/2) = \log 2 \approx 0.6931$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q08

</details>

*(この節の追加問題 `ch22-q09`, `ch22-q10`, `ch22-q11` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html で自動採点・復習できます)*

## モデルの対数確率を計算する

本物のLLMで、答え全体の対数確率を計算する関数を作ります。第20章の4択クイズで使った方法と同じです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(name)
ref_model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device).eval()

@torch.no_grad()
def answer_logprob(model, question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + a]).to(device)
    logprobs = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    return logprobs[len(p) - 1:].gather(1, ids[0, len(p):, None]).sum().item()

p = prefs[11]
lc, lr = answer_logprob(ref_model, p["question"], p["chosen"]), answer_logprob(ref_model, p["question"], p["rejected"])
print(f"良い答えの対数確率 {lc:.1f}  悪い答えの対数確率 {lr:.1f}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
良い答えの対数確率 -142.3  悪い答えの対数確率 -21.7
```

</details>

学習前のQwenにとっては、長くて個性的なボンゴ先生の答えより、短くて素っ気ない答えのほうが、ずっと「書きやすい」ようです。

<details><summary><b>基礎の補足：`answer_logprob` の仕組み：なぜ質問部分を除いて答えトークンだけを足すのか？</b>（クリックで開く）</summary>

DPOで必要なのは「質問 $x$ が与えられたときの、答え $y$ 全体の対数確率 $\log \pi(y \mid x)$」です。

1. `ids = [p + a]`：質問トークン列 `p` の後ろに、答え＋終了合図 `<|im_end|>` のトークン列 `a` をつなげてモデルに通します。
2. **1つずらしの対応**：位置 `t` の出力ロジットは位置 `t + 1` のトークンを予測します。したがって、答えの1トークン目（`ids[len(p)]`）を予測しているのは `logprobs[len(p) - 1]` です。
3. **答え部分だけの合計**：`gather` で正解トークンIDに対応する対数確率だけを取り出して `.sum()` で足します。質問部分はモデルが生成する対象ではないため除外します（第18章のSFT損失マスク `-100` と同じ発想です）。

</details>

#### 📝 確認問題（`ch22-q12` · Lv.3 · answer_logprob で質問部分を除き答えトークンだけを足す理由）

`answer_logprob` 関数では、`logprobs[len(p) - 1:]` のように**質問（プロンプト）部分を除いて、答えのトークンだけの対数確率**を合計しています。なぜ質問部分を含めないのですか？

- **a.** 求めたいのは質問の後に答えを出す条件付き確率であり、質問部分は生成対象ではないから
- **b.** 質問部分のトークン列をモデルへそのまま入力すると、チャットテンプレートの仕様により必ず実行時エラーになるから
- **c.** スライス記法 `len(p) - 1:` は回答文の末尾にある最後の1文字だけを取り出して確率を評価するための書き方だから
- **d.** 質問トークンの対数確率は常に正の数になるため、答えの負の対数確率と足し合わせると符号が打ち消されてしまうから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第18章のSFTで、質問部分のラベルに `-100` を入れて損失計算から除外した理由を思い出しましょう。
- **ヒント 2**：条件付き確率 $\pi(y \mid x)$ の対象は答え $y$ だけである点と、1トークン先の予測を取る添字 `len(p) - 1` の意味を確認してください。
- **正解**：`a`

**考え方**：モデルが位置 `len(p) - 1`（質問の末尾）で出力するロジットが「答えの1トークン目（添字 `len(p)`）」の予測になっています。そのため `logprobs[len(p) - 1:]` と `ids[0, len(p):]` がちょうど1対1に対応します。

1. 入力列は `p + a`（質問のあとに答えをつなげた列）。
1. 位置 `t` の出力ロジットは位置 `t + 1` のトークンを予測する（1つずらし）。
1. よって答えの開始 `ids[len(p)]` に対応する予測は `logprobs[len(p) - 1]` から始まる。
1. 答え部分だけを合計することで $\log\pi(y \mid x)$（`<|im_end|>` 含む）が得られる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q12

</details>

## TRLのDPOTrainerで学習する

DPOの学習は、TRLの `DPOTrainer` を使うと簡単です。データを「prompt・chosen・rejected」の形にして渡すだけです。メモリを節約するために、第19章のLoRAを使います（LoRAを使うと、参照モデルは「付せんを外した状態」として自動的に用意されるので、モデルを2つ読み込む必要がありません）。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": p["question"]}],
                         "chosen": [{"role": "assistant", "content": p["chosen"]}],
                         "rejected": [{"role": "assistant", "content": p["rejected"]}]} for p in prefs])
# DPO学習中はLoRAをOFFにした状態を参照モデルとして自動利用するため、学習前の ref_model はCPUへ退避してVRAMを節約
ref_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
policy = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
config = DPOConfig(output_dir="dpo-bongo", beta=0.1, learning_rate=5e-5, num_train_epochs=3,
                   per_device_train_batch_size=4, logging_steps=1, report_to="none", save_strategy="no",
                   bf16=False, fp16=torch.cuda.is_available())
trainer = DPOTrainer(model=policy, args=config, train_dataset=ds, processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
trainer.train()

Colabでは、学習中に「Step」と「Training Loss」の表が表示されます。でも、そこに出るのは損失だけです。DPOならではの値（`rewards/…`）は、学習が終わった後に、学習の記録 `trainer.state.log_history`（1ステップごとの記録の辞書を並べたリスト）から取り出して表示します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
for h in trainer.state.log_history:
    if "rewards/margins" in h and h["step"] % 4 == 1:     # 4ステップごとに表示
        print(f"ステップ {h['step']:2d}：損失 {h['loss']:.3f}  良い {h['rewards/chosen']:5.2f}  "
              f"悪い {h['rewards/rejected']:5.2f}  差 {h['rewards/margins']:5.2f}  正解率 {h['rewards/accuracies']:.2f}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ  1：損失 0.693  良い  0.00  悪い  0.00  差  0.00  正解率 0.00
ステップ  5：損失 0.612  良い  0.31  悪い -0.12  差  0.43  正解率 0.75
ステップ  9：損失 0.488  良い  0.85  悪い -0.57  差  1.42  正解率 1.00
ステップ 13：損失 0.401  良い  1.40  悪い -1.02  差  2.42  正解率 1.00
ステップ 17：損失 0.338  良い  1.71  悪い -1.31  差  3.02  正解率 1.00
ステップ 21：損失 0.302  良い  1.92  悪い -1.35  差  3.27  正解率 1.00
```

</details>

最初の損失は0.693（$\log 2$）。上で自分で書いた `dpo_loss` の「学習前（方策＝参照）」の値と同じです。TRLも、中で同じ式を計算しているのです。まだ何も学んでいないので、良い答えと悪い答えの差は0、つまり五分五分です。

> **💡 理論のポイント：学習ログの読み方**
>
> - `loss`：DPOの損失。0.693（$\log 2$）から下がっていけば順調
> - `rewards/chosen`：良い答えの暗黙の報酬（$\beta \times$ 元からの対数確率の変化）。上がっていくのが理想
> - `rewards/rejected`：悪い答えの暗黙の報酬。下がっていくのが理想
> - `rewards/margins`：その差。広がっていくほど、好みを学べている
> - `rewards/accuracies`：良い答えの報酬が悪い答えより高かったペアの割合
>
> `rewards/chosen` も一緒に下がっていく（良い答えも書きにくくなる）ことがよくあり、DPOの弱点として知られています。差が広がっていても、良い答えそのものの確率が下がりすぎると、文章がおかしくなることがあります。学習率を小さく、エポックを少なくするのが安全です。

学習後のモデルで、学習前に比べて、良い答えと悪い答えの書きやすさがどう変わったか見てみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
ref_model.to(device)
trainer.model.eval()
for name_, m in [("学習前", ref_model), ("DPO後", trainer.model)]:
    lc = answer_logprob(m, p["question"], p["chosen"])
    lr = answer_logprob(m, p["question"], p["rejected"])
    print(f"{name_}：良い答え {lc:7.1f}  悪い答え {lr:7.1f}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
学習前：良い答え  -142.3  悪い答え   -21.7
DPO後：良い答え  -118.6  悪い答え   -40.2
```

</details>

良い答えは書きやすく、悪い答えは書きにくくなりました。

<details><summary><b>基礎の補足：LoRA＋DPOの省メモリ仕組み と `rewards/*` ログの診断チェック</b>（クリックで開く）</summary>

- **なぜモデルが1つで済むのか？**：LoRA（第19章）では元の重み $W$ を凍結し、横付けした付せん $\Delta W$ だけを学習します。そのため **LoRAをONにすれば学習中モデル $\pi$、一時的にOFFにすれば元の参照モデル $\pi_{\text{ref}}$** として1つのモデルで2役をこなせます。
- **ログ診断のコツ**：
  - `loss` が `0.693`（$\log 2$）付近から下がっている → 出発点が正常！
  - `rewards/chosen` 上昇 ＆ `rewards/rejected` 下降 ＆ `rewards/margins` 拡大 → 理想的！
  - `rewards/margins` は増えているのに `rewards/chosen` も大きくマイナス → **良い答えの確率まで一緒に落ちている危険サイン**。学習率を下げる・エポック数を減らして生成文を確認しましょう。

</details>

#### 📝 確認問題（`ch22-q13` · Lv.3 · LoRA併用DPOの省メモリ仕組みと学習ログの読み方）

通常DPOは「学習モデル $\pi$」と「参照モデル $\pi_{\text{ref}}$」の2つを必要としますが、`DPOTrainer` に `peft_config=LoraConfig(...)` を渡すとモデルを1つしか読み込まずに済みます。なぜですか？

- **a.** LoRAを有効にすると良い答え `chosen` だけで勾配計算でき、悪い答え `rejected` のデータを読み込まなくて済むから
- **b.** 元の重みを凍結してLoRAの付せんだけ学ぶため、ONで学習モデル・OFFで参照モデルを1台で兼用できるから
- **c.** `LoraConfig` がTransformerの層数を自動で半分に削減するため、残りのVRAM領域へ参照モデルを丸ごと複製できるから
- **d.** LoRAを併用するとDPO損失の式が自動的に書き換わり、参照モデル $\pi_{\text{ref}}$ の対数確率計算そのものを省けるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第19章で学んだ「元の教科書は書き換えず、付せん（LoRAアダプタ）だけを貼って学習する」たとえを思い出しましょう。
- **ヒント 2**：ベース重みが凍結されて無傷のまま残っているなら、付せんをオンにした状態と一時的にオフにした状態がそれぞれ何に相当するか考えてください。
- **正解**：`b`

**考え方**：フルパラメータ学習では元の重み自体が書き換わるため参照モデルの丸ごとコピー（VRAM 2倍）が必要ですが、LoRAならベース重みが無傷のまま残るため、アダプタのON/OFFだけで2役をこなせます。

1. LoRA ON：ベース重み $W + \Delta W$ ＝ 学習中のモデル $\pi$。
1. LoRA OFF：ベース重み $W$ ＝ 学習前の参照モデル $\pi_{\text{ref}}$。
1. モデルのコピーが不要になり、無料Colab（T4）でもDPOが実行できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q13

</details>

#### 📝 確認問題（`ch22-q14` · Lv.4 · DPOの弱点：rewards/chosen も下がる現象の診断と対策）

あるDPO学習のログを見ると、`loss` は順調に下がり `rewards/margins` も `+3.0` まで増えましたが、`rewards/chosen` が `-2.5`、`rewards/rejected` が `-5.5` と**両方とも大きくマイナス**になり、生成される文章がおかしくなりました。何が起きており、どう対策すべきですか？

- **a.** 学習中にGPUの演算メモリが物理的に故障して符号が反転した現象であり、別のハードウェアへ買い替える以外に解決策はない
- **b.** 差さえ広げれば損失が下がるため良い答えの確率まで一緒に低下した。学習率やエポック数を抑え、SFT項併用等で対策する
- **c.** `rewards/margins` が `+3.0` と十分プラスなので学習は完全に順調であり、そのまま100エポック続ければ文章も自然に整う
- **d.** データセット内の `chosen` と `rejected` の列がすべて逆向きに入っているのが唯一の原因なので、ラベルを反転すれば直る

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：DPO損失の式が評価しているのは「$\Delta_c$ と $\Delta_r$ の絶対値」なのか、それとも「2つの相対的な差」なのかを見直しましょう。
- **ヒント 2**：「$\Delta_c = +1, \Delta_r = -2$」と「$\Delta_c = -2.5, \Delta_r = -5.5$」ではマージンが同じになる点から、何が起きているかと対策を考えてください。
- **正解**：`b`

**考え方**：DPOの目的関数は相対差 $\Delta_c - \Delta_r$ しか評価しないため、良い答えの確率まで下げつつ悪い答えをそれ以上に下げることでも損失を最小化できてしまいます。そのため `rewards/chosen` の推移と実際の生成文を必ず確認します。

1. `-2.5 - (-5.5) = +3.0` なので、マージン上は成功しているように見える。
1. しかし `rewards/chosen = -2.5 < 0` は「良い答えすら元のモデルより書きにくくなった」ことを意味する。
1. 対策：学習率を小さくする（$10^{-6}$〜$10^{-5}$台）、エポック数を抑える、あるいはCPO/ORPO/SFT項併用などを検討する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q14

</details>

> **🔬 やってみよう：学習していない質問で比べる**
>
> `bongo_test.jsonl` の質問（第18章）を、学習前のモデルとDPO後のモデルに答えさせて比べてみましょう。29組だけなので、ボンゴ先生の口調まで完全には身につかないかもしれません。でも、冷たい答えや失礼な答えは減っているはずです。第20章のLLM審査で、どちらが好まれるかを測ってみるのも良い練習です。

## アラインメントと安全性

> **💡 理論のポイント：アラインメント**
>
> LLMを「人間の意図や価値観に沿ったふるまい」に合わせることを、**アラインメント**（alignment、整合）と言います。役に立つこと（helpful）、正直であること（honest）、害を与えないこと（harmless）の3つがよく目標に挙げられます。
>
> - 危険な作り方を聞かれたら断る、差別的な発言をしない、といった**安全性**も、好みのデータ（「断る答え」をchosenにする）で学習させます
> - 人間の代わりに、原則（憲法）を与えたAIに好みを判定させる方法（**RLAIF**、Constitutional AI）もあります
> - 一方で、うまい言い回しで安全の仕組みをすり抜けようとする攻撃（**ジェイルブレイク**）もあり、守る側と破る側のいたちごっこが続いています
>
> アラインメントは、技術だけでなく「誰の好みに合わせるのか」という社会的な問いも含む、とても大切な研究分野です。

> **💡 理論のポイント：推論モデルへの道：答えが確かめられる問題で強化学習**
>
> 最近の「考えてから答える」**推論モデル**（reasoning model）は、数学やプログラミングのように**答えが正しいかを自動で確かめられる**問題で、強化学習をして作られています。人間の好みの代わりに、「答えが合っていたら報酬1、間違っていたら0」という**検証可能な報酬**（verifiable reward）を使うのです（RLVR）。モデルは、正解にたどり着くために、長く考える（考える過程のトークンを多く出す）ことを自分で学んでいきます。**GRPO** という、同じ問題に何通りも答えさせて、平均より良かった答えを強化する方法がよく使われます。第25章で、もう少し詳しく触れます。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **損失が最初から0.693でない**：参照モデルと学習モデルが違う状態から始まっています。LoRAを使う場合は気にしなくて大丈夫ですが、自分でモデルを2つ読み込む場合は、同じ重みから始めましょう。
> - **学習後に文章がおかしくなる**：学習率が大きすぎるか、エポックが多すぎます。DPOは $10^{-6}$〜$10^{-5}$ 台の小さな学習率から試すのが安全です（今回はLoRAなので少し大きめ）。
> - **`CUDA out of memory`**：DPOは、1つのペアで良い答えと悪い答えの両方を計算するので、SFTよりメモリを使います。バッチを小さくしましょう。

#### 📝 確認問題（`ch22-q15` · Lv.3 · アラインメントの3原則・RLAIF・ジェイルブレイク・RLVR（推論モデル））

最近の「長く考えてから答える」推論モデル（reasoning model）を作る際に使われる **RLVR（検証可能な報酬による強化学習）** と、従来のRLHFの違いとして正しいものはどれですか？

- **a.** RLHFは人間の主観的好みや報酬モデルを使うが、RLVRは数学の正誤やコードのテスト通過など自動検証できる報酬を使う
- **b.** RLVRで強化学習を行うと無駄な途中計算が罰せられるため、モデルは思考過程を一切書かず即座に最終数字だけ出力するようになる
- **c.** RLVRはアラインメント済みの安全なモデルから危険な手順を無理やり引き出すための、代表的なジェイルブレイク攻撃手法である
- **d.** RLVRでは人間の好みの比較ペアを必ず1億件以上手作業で作成しなければならず、RLHFより人手ラベルのコストがはるかに高い

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：RLVR の VR は **Verifiable Rewards（検証可能な報酬）** の略です。どんなタスクなら報酬を機械的に検証できるでしょう。
- **ヒント 2**：人間の主観的な好み（丁寧さなど）に頼るRLHFと、数学の正解一致やテスト通過を自動採点するRLVRの違いを比べてください。
- **正解**：`a`

**考え方**：RLVRでは自動検証できる報酬を使うことで、報酬ハッキングが起きにくく、モデルが試行錯誤を通じて自己検証や長い思考手順を自発的に獲得できることが分かりました（第25章のGRPOにつながります）。

1. アラインメントの3目標：Helpful（役に立つ）・Honest（正直）・Harmless（無害）。
1. RLHF / RLAIF / DPO：丁寧さや安全性など「主観的な良さ」を好みデータから学ぶ。
1. RLVR（GRPOなど）：数学・コードなど「自動採点できる正誤」を報酬にして推論力を伸ばす。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q15

</details>

#### 📝 確認問題（`ch22-q16` · Lv.5 · 【総合】RLHF→DPOの導出の肝と、SFT・DPO・RLVRの使い分け）

① RLHFのKL制約付き報酬最大化から、報酬モデルを消去してDPOの損失関数を導ける数学的なカギ（仮定・簡略化を含む）を自分の言葉で説明してください。② さらに「口調や出力形式を教える」「冷たい・危険な応答を減らす」「数学の計算手順を強くする」の3タスクそれぞれに、SFT・DPO・RLVRのどれを主軸に選ぶべきか理由とともに答えてください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①ではKL制約付きRLHFの最適解から報酬を逆算すると計算困難な項がどうなるかに着目し、②では各タスクで得られる教師信号の形を整理しましょう。
- **ヒント 2**：①は「最適方策の閉じた式からの逆算」「ブラッドリー・テリー差での正規化定数 $Z(x)$ の相殺」「前提となる仮定」、②はSFT（模倣）・DPO（比較好み）・RLVR（自動検証）の対応理由をまとめます。
- **解答の目安**：① RLHFのKL制約付き最適化問題の厳密な最適方策が $\pi^*(y|x) = \frac{1}{Z(x)}\pi_{\text{ref}}(y|x)\exp(r(x,y)/\beta)$ という解析解（閉じた形）で書けることを使い、逆に報酬を $r(x,y) = \beta\log(\pi^*/\pi_{\text{ref}}) + \beta\log Z(x)$ と表せる点がカギです。これをブラッドリー・テリーモデルの差 $r_c - r_r$ に代入すると、計算できない正規化定数 $\beta\log Z(x)$ が相殺されて消えるため、報酬モデルなしで直接 $\pi$ を最適化できます（ただし、好みがブラッドリー・テリーモデルに従い、オフラインの固定データ上で最適方策と報酬が1対1対応するという仮定・簡略化の上での導出です）。② 口調や出力形式の習得にはお手本を直接模倣するSFT、冷たい・危険な応答を抑えて好みに沿わせるには良い答えと悪い答えを対比できるDPO、数学の推論力強化には正誤を自動検証できるRLVR（GRPOなど）を主軸にするのが適切です。

**考え方**：現代のLLMの事後学習（Post-training）は、SFTで基礎作法を入れ、DPO/RLHFで好みと安全性を整え、RLVRで推論力を磨くという多段階パイプラインになっています。

1. KL制約付きRLHFの最適解から報酬を $\beta\log(\pi/\pi_{\text{ref}}) + \beta\log Z(x)$ と逆算する。
1. ブラッドリー・テリーモデルの差 $r_c - r_r$ に入れると、厄介な $\beta\log Z(x)$ が消えてDPO損失になる。
1. SFT（模倣）・DPO（比較の好み・安全）・RLVR（自動検証できる推論）をタスクの性質で使い分ける。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html#ex-ch22-q16

</details>

**✅ できたらチェック**

- [ ] 好みのペア（chosen / rejected）のデータの形を説明できる
- [ ] シグモイド関数と、報酬の差から確率を出す考え方を説明できる
- [ ] RLHF（報酬モデル→強化学習＋KLペナルティ）の流れを説明できる
- [ ] DPOの損失を自分で実装し、最初が $\log 2$ になることを確かめた
- [ ] TRLのDPOTrainerで学習させ、ログの `rewards/margins` を読んだ
- [ ] アラインメントと推論モデル（検証可能な報酬）の考え方を説明できる

> **📝 まとめ**
>
> - 好みの学習は「**2つを比べた判定**」から学ぶ。お手本を書くより集めやすく、細かいニュアンスを伝えられる
> - **RLHF**：報酬モデル（ブラッドリー・テリー）＋強化学習（PPO）＋KLペナルティ
> - **DPO**：報酬モデルなしで、好みのペアから直接学習。損失 $= -\log\sigma(\beta[\Delta_c - \Delta_r])$、最初は $\log 2$
> - **報酬ハッキング**に注意。アラインメントは技術と社会の両方の問題
> - **推論モデル**は、答えを確かめられる問題での強化学習（検証可能な報酬、GRPO）で作られる

**ボンゴ先生**：キミのLLMは、知識を持ち（事前学習）、答え方を覚え（SFT）、好みを知った（DPO）。次は、手足を与える番だ。計算機を使ったり、資料を検索したり、自分で考えて道具を選ぶ **エージェント** を作ろう。第1章で計算を間違えたLLMに、電卓を渡すときが来たよ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/22-alignment.html
- **次の章**：[第23章 道具を使わせる：エージェント](https://sakunokyumei.github.io/llm-research-textbook/23-agents.html)